# Objective 2.2: Multiple Linear Regression for Match Goals Scored
**Role 3 Technical Deliverable**  
**Dataset**: `lr22_team_match_208.csv` ($N = 208$)  
**Target**: `goals_scored` (Team goals scored per match after extra time, excluding shoot-outs)

### Methodological Constraints & Pre-Match Integrity:
1. All explanatory variables are computed strictly prior to kick-off.
2. Maximum 4 conceptual overlaps with LR 2.1 (FIFA Rank, Squad Value, Host Nation, Stage).
3. Model selection via exhaustive subset search (AIC/BIC) and 10-fold cross-validation.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# Visualization setup
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted')

# Load dataset
df = pd.read_csv('lr22_team_match_208.csv')
print('Dataset Shape:', df.shape)
df.head()

In [2]:
# Target Variable Inspection
y = df['goals_scored']
print('--- Target Variable Summary (goals_scored) ---')
print(y.describe())
print(f'Zero goals count: {(y == 0).sum()} ({(y == 0).mean():.1%})')
print(f'Sample Mean: {y.mean():.4f} | Sample Variance: {y.var():.4f}')

In [3]:
# Full Model & VIF Multicollinearity Diagnostics
candidate_features = [
    'team_gf_pg', 'opp_ga_pg', 'opp_gf_pg',
    'team_log_rank', 'opp_log_rank', 'team_log_value',
    'is_host_home', 'is_knockout'
]
X_full = df[candidate_features]
X_full_const = sm.add_constant(X_full)

# Calculate VIF
vif_records = []
for i, col in enumerate(candidate_features):
    vif_val = variance_inflation_factor(X_full_const.values, i + 1)
    vif_records.append({'Feature': col, 'VIF': vif_val})
vif_df = pd.DataFrame(vif_records).sort_values(by='VIF', ascending=False)
print('--- Multicollinearity (VIF) ---')
print(vif_df.to_string(index=False))

# Full OLS fit
full_ols = sm.OLS(y, X_full_const).fit()
print(full_ols.summary())

In [4]:
# Best-Subset Selection & 10-Fold CV Validation
models_dict = {
    'Full (k=8)': candidate_features,
    'Optimal AIC (k=6)': ['team_gf_pg', 'opp_ga_pg', 'team_log_rank', 'opp_log_rank', 'team_log_value', 'is_host_home'],
    'Parsimonious (k=4)': ['opp_ga_pg', 'opp_log_rank', 'team_log_value', 'is_host_home'],
    'Core Quality (k=3)': ['opp_log_rank', 'team_log_value', 'is_host_home']
}

kf = KFold(n_splits=10, shuffle=True, random_state=42)
cv_summary = []

for name, feats in models_dict.items():
    X = sm.add_constant(df[feats])
    fit_m = sm.OLS(y, X).fit()
    cv_scores = []
    for tr, te in kf.split(df):
        m_tr = sm.OLS(y.iloc[tr], sm.add_constant(df.iloc[tr][feats])).fit()
        preds = m_tr.predict(sm.add_constant(df.iloc[te][feats]))
        cv_scores.append(np.sqrt(mean_squared_error(y.iloc[te], preds)))
    cv_summary.append({
        'Model': name,
        'k': len(feats),
        'R2': fit_m.rsquared,
        'Adj_R2': fit_m.rsquared_adj,
        'AIC': fit_m.aic,
        'BIC': fit_m.bic,
        '10-Fold CV RMSE': np.mean(cv_scores)
    })

pd.DataFrame(cv_summary)

In [5]:
# Final Selected Model (k=6) & Robust HC3 Standard Errors
opt_features = ['team_gf_pg', 'opp_ga_pg', 'team_log_rank', 'opp_log_rank', 'team_log_value', 'is_host_home']
X_opt_const = sm.add_constant(df[opt_features])
final_model = sm.OLS(y, X_opt_const).fit()
final_model_hc3 = final_model.get_robustcov_results(cov_type='HC3')

summary_coefs = pd.DataFrame({
    'Coef': final_model.params,
    'OLS_StdErr': final_model.bse,
    'OLS_p': final_model.pvalues,
    'HC3_StdErr': final_model_hc3.bse,
    'HC3_p': final_model_hc3.pvalues
})
print('--- Classical OLS vs HC3 Robust Standard Errors ---')
print(summary_coefs.round(4))
print(final_model.summary())

In [6]:
# Regression Diagnostic Suite
residuals = final_model.resid
fitted = final_model.fittedvalues

# Diagnostic tests
shapiro_w, shapiro_p = stats.shapiro(residuals)
jb_stat, jb_p, skew, kurt = sm.stats.jarque_bera(residuals)
bp_stat, bp_p = het_breuschpagan(residuals, final_model.model.exog)[:2]
dw_stat = durbin_watson(residuals)

print(f'Shapiro-Wilk W: {shapiro_w:.4f} (p = {shapiro_p:.4e})')
print(f'Jarque-Bera JB: {jb_stat:.4f} (p = {jb_p:.4e}, Skewness = {skew:.3f}, Kurtosis = {kurt:.3f})')
print(f'Breusch-Pagan LM: {bp_stat:.4f} (p = {bp_p:.4f})')
print(f'Durbin-Watson DW: {dw_stat:.4f}')